# 17 — Baseline Robot Güçlendirme Deneyleri

Bu notebook üretimde kullanılan Baseline Robot kurallarını değiştirmez. Mevcut `AL` sinyallerini dört farklı bilgi türüyle filtreleyen araştırma varyantlarını kontrollü biçimde karşılaştırır:

- BIST100'e göre göreceli güç
- EMA50'ye ATR bazlı aşırı uzaklık
- BIST100 piyasa genişliği
- EMA50 eğimi

Metodoloji:

```text
Development (2018–2022)
→ her aileden tek eşik seçimi

Validation (2023–2024)
→ aile kazananları ve kombinasyonlar için kabul/ret

Zaman blokları
→ dönem dayanıklılığı

Audit (2025+)
→ yalnızca karar kilitlendikten sonra, varsayılan olarak kapalı
```

Bir varyant kabul edilse bile günlük sinyal sistemi otomatik olarak değiştirilmez.

In [ ]:
from dataclasses import asdict
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src").is_dir()
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

%load_ext autoreload
%autoreload 2

from src.features import add_indicators
from src.signals import build_market_regime
from src.presets import (
    FINAL_STRATEGY_CONFIG,
    FINAL_PORTFOLIO_CONFIG,
)
from src.baseline_enhancements import (
    baseline_variant,
    default_single_factor_variants,
    build_combination_variants,
    prepare_enhancement_dataset,
    run_enhancement_grid,
    select_development_family_winners,
    build_validation_acceptance_table,
    summarize_block_stability,
    variants_by_name,
    evaluate_enhancement_variant,
    save_enhancement_artifacts,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

## 1. Araştırma verisini yükle

Bu notebook `data/processed` altındaki tam geçmiş veriyi kullanır. `data/live` günlük uygulama içindir ve bu araştırmada kullanılmaz.

In [ ]:
stock_prices = pd.read_parquet(
    PROJECT_ROOT
    / "data"
    / "processed"
    / "bist100_robot_clean.parquet"
)

market_prices = pd.read_parquet(
    PROJECT_ROOT
    / "data"
    / "processed"
    / "xu100_robot_clean.parquet"
)

stock_features = add_indicators(stock_prices)
market_features = add_indicators(market_prices)
market_regime = build_market_regime(
    market_features
)

prepared_prices = prepare_enhancement_dataset(
    stock_features=stock_features,
    market_features=market_features,
    market_regime=market_regime,
    strategy_config=FINAL_STRATEGY_CONFIG,
)

print("Hazırlanan veri:", prepared_prices.shape)
print(
    "Tarih aralığı:",
    prepared_prices["Date"].min(),
    "→",
    prepared_prices["Date"].max(),
)

display(
    prepared_prices[
        [
            "Date",
            "Ticker",
            "Score",
            "Baseline_Signal",
            "RS_63",
            "RS_126",
            "EMA50_DISTANCE_ATR",
            "BREADTH_EMA50",
            "EMA50_SLOPE_10",
        ]
    ].tail()
)

## 2. Development — tek faktör deneyleri

Eşikler önceden tanımlanmıştır. Development sonucu görülmeden yeni eşik eklememek, aşırı uyum riskini azaltır.

In [ ]:
DEVELOPMENT_PERIOD = {
    "Development": (
        "2018-01-01",
        "2022-12-31",
    )
}

single_variants = default_single_factor_variants()

development_results = run_enhancement_grid(
    prepared_prices=prepared_prices,
    variants=single_variants,
    strategy_config=FINAL_STRATEGY_CONFIG,
    portfolio_config=FINAL_PORTFOLIO_CONFIG,
    periods=DEVELOPMENT_PERIOD,
)

development_columns = [
    "Variant",
    "Family",
    "CAGR_%",
    "Max_Drawdown_%",
    "Profit_Factor",
    "Sharpe",
    "Calmar",
    "Trade_Count",
    "Signal_Pass_Rate_%",
]

display(
    development_results[
        development_columns
    ].sort_values(
        ["Calmar", "CAGR_%"],
        ascending=False,
    )
)

## 3. Development aile kazananları

Her aileden en fazla bir eşik seçilir. Seçim sadece Development dönemine bakılarak yapılır.

In [ ]:
development_winners = (
    select_development_family_winners(
        development_results
    )
)

winner_columns = [
    "Variant",
    "Family",
    "CAGR_%",
    "CAGR_Delta_pp",
    "Max_Drawdown_%",
    "Max_DD_Improvement_pp",
    "Profit_Factor",
    "Calmar",
    "Trade_Count",
    "Trade_Fraction",
]

display(development_winners[winner_columns])

single_map = variants_by_name(
    single_variants
)

family_winner_variants = [
    single_map[name]
    for name in development_winners[
        "Variant"
    ].tolist()
]

combination_variants = (
    build_combination_variants(
        family_winner_variants,
        minimum_size=2,
        maximum_size=3,
    )
)

validation_variants = (
    [baseline_variant()]
    + family_winner_variants
    + combination_variants
)

print(
    "Validation aday sayısı:",
    len(validation_variants),
)
print(
    *[variant.name for variant in validation_variants],
    sep="\n- ",
)

## 4. Validation — kabul veya ret

Kombinasyonlar, Validation sonucu görülmeden Development aile kazananlarından oluşturulur.

In [ ]:
VALIDATION_PERIOD = {
    "Validation": (
        "2023-01-01",
        "2024-12-31",
    )
}

validation_results = run_enhancement_grid(
    prepared_prices=prepared_prices,
    variants=validation_variants,
    strategy_config=FINAL_STRATEGY_CONFIG,
    portfolio_config=FINAL_PORTFOLIO_CONFIG,
    periods=VALIDATION_PERIOD,
)

acceptance_table = (
    build_validation_acceptance_table(
        validation_results
    )
)

acceptance_columns = [
    "Variant",
    "Family",
    "CAGR_%",
    "CAGR_Delta_pp",
    "Max_Drawdown_%",
    "Max_DD_Improvement_pp",
    "Profit_Factor",
    "Profit_Factor_Delta",
    "Sharpe",
    "Calmar",
    "Calmar_Improvement_%",
    "Trade_Count",
    "Trade_Fraction",
    "Return_Case_Accepted",
    "Risk_Case_Accepted",
    "Enhancement_Accepted",
]

display(
    acceptance_table[
        acceptance_columns
    ]
)

### Kabul kuralları

**Getiri odaklı kabul**

```text
Validation CAGR farkı       >= +1,5 yüzde puan
Drawdown                    kötüleşmemeli
Calmar iyileşmesi           >= %5
Profit Factor               düşmemeli
İşlem sayısı                Baseline'ın en az %70'i
```

**Risk odaklı kabul**

```text
Validation CAGR farkı       >= -1,0 yüzde puan
Drawdown iyileşmesi         >= 2 yüzde puan
Calmar iyileşmesi           >= %8
Profit Factor               Baseline'ın en az %95'i
İşlem sayısı                Baseline'ın en az %70'i
```

In [ ]:
accepted_names = acceptance_table.loc[
    acceptance_table[
        "Enhancement_Accepted"
    ],
    "Variant",
].tolist()

validation_map = variants_by_name(
    validation_variants
)

accepted_variants = [
    validation_map[name]
    for name in accepted_names
]

print(
    "Validation kabul edilen varyantlar:",
    accepted_names or "Yok",
)

## 5. Zaman bloğu dayanıklılığı

Sadece Validation kabul edilen adaylar çalıştırılır. Adayın en az üç blokta risk ayarlı katkı sağlaması beklenir.

In [ ]:
STABILITY_PERIODS = {
    "2019-2020": (
        "2019-01-01",
        "2020-12-31",
    ),
    "2021-2022": (
        "2021-01-01",
        "2022-12-31",
    ),
    "2023": (
        "2023-01-01",
        "2023-12-31",
    ),
    "2024": (
        "2024-01-01",
        "2024-12-31",
    ),
}

block_results = None
block_summary = None

if accepted_variants:
    block_results = run_enhancement_grid(
        prepared_prices=prepared_prices,
        variants=(
            [baseline_variant()]
            + accepted_variants
        ),
        strategy_config=FINAL_STRATEGY_CONFIG,
        portfolio_config=FINAL_PORTFOLIO_CONFIG,
        periods=STABILITY_PERIODS,
    )

    block_summary = summarize_block_stability(
        block_results
    )

    display(block_summary)
else:
    print(
        "Validation kabul edilen aday olmadığı için "
        "blok analizi çalıştırılmadı."
    )

## 6. Validation görsel karşılaştırması

In [ ]:
plot_data = acceptance_table.loc[
    acceptance_table["Variant"].ne(
        "Baseline"
    )
].copy()

plt.figure(figsize=(12, 7))
plt.scatter(
    plot_data["Max_Drawdown_%"],
    plot_data["CAGR_%"],
    s=(
        plot_data["Trade_Fraction"]
        .fillna(0)
        .clip(lower=0)
        * 180
    ),
)

for row in plot_data.itertuples():
    plt.annotate(
        row.Variant,
        (
            row._asdict()["Max_Drawdown_%"],
            row._asdict()["CAGR_%"],
        ),
        fontsize=8,
        xytext=(4, 4),
        textcoords="offset points",
    )

baseline_row = acceptance_table.loc[
    acceptance_table["Variant"].eq(
        "Baseline"
    )
].iloc[0]

plt.axhline(
    baseline_row["CAGR_%"],
    linestyle="--",
    label="Baseline CAGR",
)
plt.axvline(
    baseline_row["Max_Drawdown_%"],
    linestyle="--",
    label="Baseline Drawdown",
)

plt.title(
    "Validation — CAGR ve Maksimum Drawdown"
)
plt.xlabel("Maksimum Drawdown (%)")
plt.ylabel("CAGR (%)")
plt.legend()
plt.tight_layout()
plt.show()

## 7. Kabul edilen en iyi adayın equity karşılaştırması

Bu grafik yalnızca Validation karşılaştırmasıdır; üretim stratejisi değişikliği değildir.

In [ ]:
best_candidate_name = None
best_candidate_equity = None
baseline_equity = None

accepted_table = acceptance_table.loc[
    acceptance_table[
        "Enhancement_Accepted"
    ]
].copy()

if not accepted_table.empty:
    best_candidate_name = accepted_table.iloc[
        0
    ]["Variant"]
    best_candidate = validation_map[
        best_candidate_name
    ]

    _, baseline_equity, _ = (
        evaluate_enhancement_variant(
            prepared_prices=prepared_prices,
            variant=baseline_variant(),
            strategy_config=(
                FINAL_STRATEGY_CONFIG
            ),
            portfolio_config=(
                FINAL_PORTFOLIO_CONFIG
            ),
            start="2023-01-01",
            end="2024-12-31",
        )
    )

    _, best_candidate_equity, _ = (
        evaluate_enhancement_variant(
            prepared_prices=prepared_prices,
            variant=best_candidate,
            strategy_config=(
                FINAL_STRATEGY_CONFIG
            ),
            portfolio_config=(
                FINAL_PORTFOLIO_CONFIG
            ),
            start="2023-01-01",
            end="2024-12-31",
        )
    )

    plt.figure(figsize=(13, 7))
    plt.plot(
        baseline_equity["Date"],
        baseline_equity["Equity"],
        label="Baseline",
    )
    plt.plot(
        best_candidate_equity["Date"],
        best_candidate_equity["Equity"],
        label=best_candidate_name,
    )
    plt.title(
        "Validation Equity — Baseline ve En İyi Kabul Edilen Aday"
    )
    plt.xlabel("Tarih")
    plt.ylabel("Portföy Değeri (TL)")
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print(
        "Equity karşılaştırması için kabul edilen aday yok."
    )

## 8. Audit — varsayılan olarak kapalı

Önce Development, Validation ve blok sonuçlarına göre karar kilitlenmelidir. `RUN_AUDIT = True` yalnızca bundan sonra kullanılmalıdır.

> 2025+ döneminin daha önce incelendiği unutulmamalıdır; kusursuz bir holdout değildir.

In [ ]:
RUN_AUDIT = False

audit_results = None

if RUN_AUDIT:
    if block_summary is not None:
        stable_names = block_summary.loc[
            block_summary[
                "Stable_Across_Blocks"
            ],
            "Variant",
        ].tolist()
    else:
        stable_names = accepted_names

    audit_variants = [
        validation_map[name]
        for name in stable_names
    ]

    if audit_variants:
        audit_end = min(
            prepared_prices["Date"].max(),
            market_prices["Date"].max(),
        ).strftime("%Y-%m-%d")

        audit_results = run_enhancement_grid(
            prepared_prices=prepared_prices,
            variants=(
                [baseline_variant()]
                + audit_variants
            ),
            strategy_config=FINAL_STRATEGY_CONFIG,
            portfolio_config=FINAL_PORTFOLIO_CONFIG,
            periods={
                "Audit_2025_Plus": (
                    "2025-01-01",
                    audit_end,
                )
            },
        )

        display(audit_results)
    else:
        print(
            "Audit için stabil kabul edilen aday yok."
        )
else:
    print(
        "Audit kapalı. Karar kilitlenmeden açmayın."
    )

## 9. Sonuçları kaydet

Çıktılar `.gitignore` kapsamındaki `results/baseline_enhancements` klasörüne kaydedilir.

In [ ]:
OUTPUT_DIR = (
    PROJECT_ROOT
    / "results"
    / "baseline_enhancements"
)

metadata = {
    "method": (
        "Development family selection, "
        "Validation acceptance, block stability"
    ),
    "production_baseline_changed": False,
    "development_period": (
        DEVELOPMENT_PERIOD
    ),
    "validation_period": (
        VALIDATION_PERIOD
    ),
    "stability_periods": (
        STABILITY_PERIODS
    ),
    "audit_enabled": RUN_AUDIT,
    "strategy_config": asdict(
        FINAL_STRATEGY_CONFIG
    ),
    "portfolio_config": asdict(
        FINAL_PORTFOLIO_CONFIG
    ),
    "stock_data_start": (
        prepared_prices["Date"]
        .min()
        .isoformat()
    ),
    "stock_data_end": (
        prepared_prices["Date"]
        .max()
        .isoformat()
    ),
    "warning": (
        "Current-universe breadth and historical "
        "BIST100 tests may contain survivorship bias."
    ),
}

artifact_paths = save_enhancement_artifacts(
    output_directory=OUTPUT_DIR,
    development_results=development_results,
    development_winners=development_winners,
    validation_results=validation_results,
    acceptance_table=acceptance_table,
    block_results=block_results,
    block_summary=block_summary,
    audit_results=audit_results,
    metadata=metadata,
)

for name, path in artifact_paths.items():
    print(name, "→", path)

## Karar notu

Bu notebook sonunda üç olası sonuç vardır:

1. **Hiçbir varyant kabul edilmez:** Baseline korunur.
2. **Bir varyant Validation'da kabul edilir ama bloklarda stabil değildir:** Baseline korunur.
3. **Bir varyant Validation ve blok testlerini geçer:** Audit kararı kilitlendikten sonra çalıştırılır; günlük sinyal kodu yine ayrıca ve bilinçli olarak güncellenir.

Bu yapı, sadece backtest CAGR'ı yükseldiği için stratejiye yeni indikatör eklenmesini engeller.